In [14]:
!pip install chembl_webresource_client -q

In [8]:
!pip install bioservices -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 720.1/720.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.9/264.9 kB 14.4 MB/s eta 0:00:00


In [ ]:
pip install pubchempy

In [3]:
import pandas as pd
import requests
import json

import pubchempy as pcp

In [6]:
import requests
import xml.etree.ElementTree as ET
import time

def consultar_chebi(chebi_id):
    # Endpoint REST para obter a entidade completa
    url = f"https://www.ebi.ac.uk/webservices/chebi/2.0/getCompleteEntity?chebiId={chebi_id}"
    try:
        response = requests.get(url)
        time.sleep(5.0)
        response.raise_for_status() # Verifica se houve erro na requisição

        # O ChEBI retorna XML. Vamos parsear:
        root = ET.fromstring(response.content)
        # Namespace do XML da EBI
        ns = {'chebi': 'http://www.ebi.ac.uk/webservices/chebi'}
        # Extração de dados básicos
        nome = root.find('.//chebi:chebiAsciiName', ns).text
        id_final = root.find('.//chebi:chebiId', ns).text

        print(f"--- Resultado para: {nome} ({id_final}) ---")

        # Listas para organizar os papéis e classes
        papeis_biologicos = []
        classes_quimicas = []
        sub_ontologias = []

        # Buscando relações na Ontologia
        for relation in root.findall('.//chebi:OntologyRelationships', ns):
            rel_type = relation.find('chebi:type', ns).text
            rel_name = relation.find('chebi:data', ns).text

            if rel_type == 'has role':
                papeis_biologicos.append(rel_name)
            elif rel_type == 'is a':
                classes_quimicas.append(rel_name)

        # Exibindo os resultados
        print(f"\n✅ Papéis Biológicos: {', '.join(papeis_biologicos) if papeis_biologicos else 'Nenhum encontrado'}")
        print(f"✅ Classes Químicas (Is a): {', '.join(classes_quimicas) if classes_quimicas else 'Nenhuma encontrada'}")

    except Exception as e:
        print(f"Erro ao consultar o ID {chebi_id}: {e}")

# Exemplo de uso: Cafeína (CHEBI:15377)
consultar_chebi("CHEBI:15377")

Erro ao consultar o ID CHEBI:15377: 500 Server Error: Internal Server Error for url: https://www.ebi.ac.uk/webservices/chebi/2.0/getCompleteEntity?chebiId=CHEBI:15377


In [17]:
!pip install chembl_webresource_client -q

import time
from chembl_webresource_client.new_client import new_client

def consultar_chembl_v4(termo_busca):
    print(f"🔎 Analisando: {termo_busca}...")
    try:
        mol_api = new_client.molecule
        mech_api = new_client.mechanism

        # Busca por ID ChEMBL, ChEBI ou Nome
        if "CHEMBL" in termo_busca:
            res = mol_api.filter(molecule_chembl_id=termo_busca)
        elif "CHEBI:" in termo_busca:
            res = mol_api.filter(chebi_id=termo_busca)
        else:
            res = mol_api.search(termo_busca)

        if not res:
            print(f"❌ {termo_busca} não encontrado.")
            return

        # Identifica o registro pai (evita erros com sais/variantes)
        mol_ref = res[0]
        parent_id = mol_ref.get('molecule_hierarchy', {}).get('parent_chembl_id', mol_ref['molecule_chembl_id'])
        mol = mol_api.get(parent_id)

        # Nome: Tenta nome oficial, depois sinônimos
        nome = mol.get('pref_name') or (mol.get('molecule_synonyms')[0].get('molecule_synonym') if mol.get('molecule_synonyms') else "NOME DESCONHECIDO")

        # Papel Biológico: Combina códigos ATC e Mecanismos de Ação
        atc = mol.get('atc_classifications', [])
        mechs = [m.get('mechanism_of_action') for m in mech_api.filter(molecule_chembl_id=parent_id) if m.get('mechanism_of_action')]
        papeis = list(set(atc + mechs))

        # Categoria: Valida max_phase contra None para evitar o erro de comparação
        fase = mol.get('max_phase') or 0
        is_drug = mol.get('therapeutic_flag', False)

        if fase == 4 or is_drug:
            categoria = "Fármaco Aprovado"
        elif fase > 0:
            categoria = f"Fármaco em Investigação (Fase {fase})"
        else:
            categoria = "Metabólito / Composto de Pesquisa"

        print(f"\n--- {nome.upper()} ---")
        print(f"🆔 ID ChEMBL: {parent_id}")
        print(f"🧬 Papel: {', '.join(papeis) if papeis else 'Nenhum'}")
        print(f"🧪 Classe: {mol.get('molecule_type', 'N/A')}")
        print(f"📂 Status: {categoria}")
        print("-" * 45)

        time.sleep(1)

    except Exception as e:
        print(f"❌ Erro em {termo_busca}: {e}")

# Teste
meus_testes = ["CHEMBL521", "Ibuprofen", "CHEBI:15377"]
for t in meus_testes:
    consultar_chembl_v4(t)

🔎 Analisando: CHEMBL521...

--- IBUPROFEN ---
🆔 ID ChEMBL: CHEMBL521
🧬 Papel: G02CC01, C01EB16, Cyclooxygenase inhibitor, M01AE51, R02AX02, M01AE01, M02AA13
🧪 Classe: Small molecule
📂 Status: Fármaco Aprovado
---------------------------------------------
🔎 Analisando: Ibuprofen...

--- IBUPROFEN ---
🆔 ID ChEMBL: CHEMBL521
🧬 Papel: G02CC01, C01EB16, Cyclooxygenase inhibitor, M01AE51, R02AX02, M01AE01, M02AA13
🧪 Classe: Small molecule
📂 Status: Fármaco Aprovado
---------------------------------------------
🔎 Analisando: CHEBI:15377...

--- NOME DESCONHECIDO ---
🆔 ID ChEMBL: CHEMBL6329
🧬 Papel: Nenhum
🧪 Classe: Small molecule
📂 Status: Metabólito / Composto de Pesquisa
---------------------------------------------


In [16]:

import time
from chembl_webresource_client.new_client import new_client

def consultar_chembl_v3(termo_busca):
    print(f"🔎 Analisando: {termo_busca}...")

    try:
        mol_api = new_client.molecule
        mech_api = new_client.mechanism

        # 1. Busca inicial (Filtra por ID exato ou busca por nome)
        if termo_busca.startswith("CHEMBL"):
            res = mol_api.filter(molecule_chembl_id=termo_busca)
        elif termo_busca.startswith("CHEBI:"):
            res = mol_api.filter(chebi_id=termo_busca)
        else:
            res = mol_api.search(termo_busca)

        if not res:
            print(f"❌ {termo_busca} não encontrado.")
            return

        # Pegamos o primeiro match
        mol_inicial = res[0]

        # 2. TRUQUE DO PAI: Se a molécula for um derivado, buscamos o ID do PAI
        # Isso garante que pegaremos o nome correto e o status de fármaco
        parent_id = mol_inicial.get('molecule_hierarchy', {}).get('parent_chembl_id', mol_inicial['molecule_chembl_id'])

        # Se for diferente, refazemos a busca pelo registro principal
        if parent_id != mol_inicial['molecule_chembl_id']:
            mol = mol_api.get(parent_id)
        else:
            mol = mol_inicial

        # --- EXTRAÇÃO DE DADOS ---

        # Nome (Tenta nome preferencial, se não, tenta o primeiro sinônimo)
        nome = mol.get('pref_name')
        if not nome and mol.get('molecule_synonyms'):
            nome = mol['molecule_synonyms'][0].get('molecule_synonym')
        nome = nome or "NOME DESCONHECIDO"

        # Papel Biológico 1: ATC Classifications
        atc_codes = mol.get('atc_classifications', [])

        # Papel Biológico 2: Mecanismo de Ação (Alvos biológicos)
        mechs = mech_api.filter(molecule_chembl_id=parent_id)
        mecanismos = [m.get('mechanism_of_action') for m in mechs if m.get('mechanism_of_action')]

        papeis = list(set(atc_codes + mecanismos))

        # Classe Química
        classe = mol.get('molecule_type', 'Small molecule')

        # Sub-ontologia (Lógica de Fármaco vs Metabólito)
        fase = mol.get('max_phase', 0)
        is_drug = mol.get('therapeutic_flag', False)

        if fase == 4 or is_drug:
            categoria = "Fármaco Aprovado"
        elif fase > 0:
            categoria = f"Fármaco em Investigação (Fase {fase})"
        else:
            categoria = "Metabólito / Composto de Pesquisa"

        # --- OUTPUT ---
        print(f"\n--- {nome.upper()} ---")
        print(f"🆔 ID ChEMBL: {parent_id}")
        print(f"🧬 Papel Biológico: {', '.join(papeis) if papeis else 'Papel não mapeado'}")
        print(f"🧪 Classe Química: {classe}")
        print(f"📂 Categoria: {categoria}")
        print("-" * 45)

        time.sleep(1)

    except Exception as e:
        print(f"❌ Erro ao processar {termo_busca}: {e}")

# Teste com os IDs que deram erro antes
meus_testes = ["CHEMBL521", "Ibuprofen", "CHEBI:15377"]

for t in meus_testes:
    consultar_chembl_v3(t)

🔎 Analisando: CHEMBL521...

--- IBUPROFEN ---
🆔 ID ChEMBL: CHEMBL521
🧬 Papel Biológico: G02CC01, C01EB16, Cyclooxygenase inhibitor, M01AE51, R02AX02, M01AE01, M02AA13
🧪 Classe Química: Small molecule
📂 Categoria: Fármaco Aprovado
---------------------------------------------
🔎 Analisando: Ibuprofen...

--- IBUPROFEN ---
🆔 ID ChEMBL: CHEMBL521
🧬 Papel Biológico: G02CC01, C01EB16, Cyclooxygenase inhibitor, M01AE51, R02AX02, M01AE01, M02AA13
🧪 Classe Química: Small molecule
📂 Categoria: Fármaco Aprovado
---------------------------------------------
🔎 Analisando: CHEBI:15377...
❌ Erro ao processar CHEBI:15377: '>' not supported between instances of 'NoneType' and 'int'
